In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l2.5 Positional information

> 🎯 **Goal:** Give the model a sense of word order by attaching a position vector to each token.

**Why this matters.** The heart of a transformer is self-attention, and attention has a surprising blind spot: it treats its input as an unordered *set*. Without help, "dog bites man" and "man bites dog" look identical to it, even though they mean opposite things. Since order is most of what grammar is, the model needs order information injected explicitly. This lesson is where that information enters.

**The intuition.** Imagine a book whose pages got shuffled. If the pages have no page numbers, you can never recover the story; any order looks as plausible as any other. Now stamp a page number on each page. Even shuffled, you can reassemble the sequence. Positional encoding stamps a "page number" onto each token, so the model can tell first from second from third.

**The mechanics.** A *positional embedding* is a second lookup table, but indexed by position instead of by token id. `nn.Embedding(block, dim)` makes one row per position up to `block` (the maximum sequence length, also called the *block size* or context window). Position 0 gets one vector, position 1 gets a different vector, and so on. In the full model (unit u4) these position vectors are *added* to the token embeddings, so each token's final vector encodes both what it is and where it sits. These vectors are *learned*, just like token embeddings.

In [ ]:
from torch import nn

block, dim = 16, 8
pos_emb = nn.Embedding(block, dim)
p0 = pos_emb(torch.tensor(0))
p1 = pos_emb(torch.tensor(1))
print("position 0 and 1 get different vectors:", not torch.equal(p0, p1))

**What just happened.** We looked up the vectors for position 0 and position 1 and confirmed they differ. That difference is the whole mechanism: because each slot in the sequence carries a distinct vector, the model can distinguish "this token came first" from "this token came second" even though attention itself ignores order.

In the real model these position vectors are added onto the token embeddings before the first attention layer, so order rides along for free through the rest of the network.

⚠️ **Common pitfalls**
- Sequence longer than `block`: this table only has rows for positions 0 up to `block - 1`. Feed a sequence longer than the block size and the lookup runs off the end and errors. The block size caps how much context the model can see.
- Forgetting to add them: if you compute position embeddings but never add them to the token embeddings, the model stays order-blind. The vectors only help once they are actually combined with the tokens.

🏋️ **Try it yourself.** Print the vectors for positions 0, 1, and 2 and eyeball how different they are. Then look up `pos_emb(torch.tensor(20))` with `block` still 16 and watch it raise an index error, making the block-size limit concrete. (Note: unit u6 later swaps these learned vectors for RoPE, a smarter relative scheme, but the goal stays the same.)

In [ ]:
assert not torch.equal(p0, p1)